In [5]:
# # --- Instalacja / naprawa środowiska (URUCHOM JAKO PIERWSZE) ---
# # Wymóg: wszystkie install/uninstall są w tej komórce.
# # Target: Google Colab (Python 3.12) + T4, bez 4-bit (USE_4BIT=False).

# import re
# import sys
# import time
# import subprocess

# install_start = time.time()
# PY = sys.executable


# def run(cmd: list[str]) -> str:
#     return subprocess.check_output(cmd, text=True).strip()


# def get_torch_version() -> str | None:
#     try:
#         out = run([PY, "-c", "import torch; print(torch.__version__)"])
#         return out
#     except Exception:
#         return None


# def major_minor(v: str | None):
#     m = re.match(r"^(\d+)\.(\d+)", v or "")
#     return (int(m.group(1)), int(m.group(2))) if m else (0, 0)


# print("python:", sys.version.split()[0])
# print("torch(before):", get_torch_version())

# # 1) Napraw torch jeśli jest zbyt stary dla transformers (wymagane: torch>=2.4)
# if major_minor(get_torch_version()) < (2, 4):
#     print("[FIX] torch jest zbyt stary (wymagane torch>=2.4) -> reinstalacja torch/vision/audio (cu121)")
#     # UWAGA: używamy *tego samego* interpretera co kernel
#     !{PY} -m pip -q uninstall -y torch torchvision torchaudio
#     !{PY} -m pip -q install --index-url https://download.pytorch.org/whl/cu121       torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1

#     print("torch(after pip):", get_torch_version())
#     print("[IMPORTANT] Teraz zrób Runtime/Kernel -> Restart i uruchom notebook od początku.")
#     raise SystemExit("Restart required after torch reinstall")

# # torchao: PEFT może próbować go użyć; wymagane torchao>=0.16
# !{PY} -m pip -q install -U "torchao>=0.16.0"

# # 2) Reszta zależności (nie ruszamy torch)
# # Uwaga: Colab z RAPIDS/cudf wymaga pandas<2.4. Pinujemy stabilnie do 2.2.x.
# !{PY} -m pip -q install -U --upgrade-strategy only-if-needed   "transformers"   "accelerate"   "datasets"   "huggingface_hub"   "peft"   "sentencepiece"   "scikit-learn"   "pandas==2.2.2"   "matplotlib"

# # 3) Naprawa ABI pyarrow (tylko jeśli wcześniej środowisko było rozjechane)
# !{PY} -m pip -q install -U --force-reinstall "pyarrow"

# # 4) 4-bit OFF: nie instalujemy bitsandbytes.

# print("Install done in", round(time.time() - install_start, 1), "s")
# print("torch(final):", get_torch_version())


In [6]:
#@title Dane studenta

import requests

Student_ID = "512751"  #@param {type:"string"}
Mail = "alekra28@st.amu.edu.pl"  #@param {type:"string"}
Grupa = ""  #@param {type:"string"}
Link_do_tego_notebooka = "https://github.com/senketsutsu/ZZTAI"  #@param {type:"string"}

def wyslij_odpowiedz(task_id, final_answer):
    final_answer = str(final_answer)
    final_answer_size = len(final_answer)
    final_answer_send = final_answer[:500] + "\n" + str(final_answer_size) + " znaków"
    data = {
        "student_id": Student_ID,
        "student_mail": Mail,
        "task": task_id,
        "grupa": Grupa,
        "answer": final_answer_send,
        "share_link": Link_do_tego_notebooka,
    }
    url = "https://www.duszekjk.com/zztai/api/submit_answer/"
    r = requests.post(url, json=data, timeout=20)
    print(r.status_code)
    try:
        j = r.json()
        if "points" in j:
            print("Punkty:", j["points"])
        else:
            print(j)
    except Exception:
        print(r.text[:1000])


# Lekcja E — LoRA/QLoRA dla małego LLM + augmentacja danych tekstowych do klasyfikacji


# Organizacja zadań

Ta lekcja jest „analogiem” Lekcji D, ale zamiast Stable Diffusion używamy **małego LLM**.

Pipeline (intuicyjnie):
1) bierzemy **mały, etykietowany** zbiór tekstowy,
2) trenujemy **adapter LoRA/QLoRA** na LLM, żeby nauczył się generować tekst w stylu danych + zgodny z etykietą,
3) generujemy **syntetyczne próbki** (np. 50× więcej),
4) trenujemy klasyfikator „przed” i „po” augmentacji i porównujemy metryki.

Zadania:
- `E01` Przygotowanie małego zbioru wejściowego + podział `train/val/test`
- `E02` Trening adaptera LoRA/QLoRA (LLM „po dostrojeniu”)
- `E03` Generowanie syntetycznego zbioru (ok. 50× większego)
- `E04` Automatyczne filtrowanie i sanity-check syntetyków
- `E05` Budowa finalnego zbioru do klasyfikacji
- `E06` Klasyfikator i porównanie dokładności (przed/po augmentacji)

Odpowiedzi końcowe:
- `E21` — ścieżka do folderu z adapterem LoRA (na Google Drive),
- `E61` — accuracy na zbiorze test „real-only”,
- `E62` — accuracy na zbiorze test po augmentacji,
- `E63` — różnica `E62 - E61`.

Co jest **wymagane do oceny**, a co jest **opcjonalne**:
- wymagane do oceny: tylko `E21`, `E61`, `E62`, `E63`,
- opcjonalne (extra, nie jest oceniane automatem): upload do Hugging Face i uruchomienie na iPhone.

Nie trzeba oddawać ścieżek do CSV, seedów ani parametrów generacji — ale notebook je zapisuje,
żeby eksperyment dało się powtórzyć.


## Autoryzacja i wymagania techniczne (Colab)

W tej lekcji:
- używamy modeli z Hugging Face (HF),
- opcjonalnie zapisujemy wyniki na Google Drive.

Co trzeba zrobić:
1. Zaloguj się do Hugging Face:
   - do samego *pobrania* modelu wystarczy token `read`,
   - jeśli chcesz skorzystać z sekcji uploadu (extra), potrzebujesz tokena z prawem zapisu (`write`).
2. (Opcjonalnie) Zamontuj Google Drive, żeby wyniki przetrwały restart Colab.

Oficjalne źródła (warto mieć pod ręką):
- HF tokens: https://huggingface.co/docs/hub/security-tokens
- `hf auth login`: https://huggingface.co/docs/huggingface_hub/main/package_reference/cli#hf-auth-login
- Transformers: https://huggingface.co/docs/transformers
- PEFT (LoRA/QLoRA): https://huggingface.co/docs/peft
- Datasets: https://huggingface.co/docs/datasets

Uwaga o powtarzalności:
- **augmentacja** zwykle używa sampling (bo ma „zwiększać różnorodność”), ale robimy ją z ustalonym `SEED`
  i zapisujemy wynik do CSV, żeby dało się porównać modele „przed/po”,
- **odpowiedzi do automatu** (`E61–E63`) są deterministyczne, bo liczymy metryki na zapisanych danych,
- jeśli generujesz pojedyncze przykłady do prezentacji, ustaw `temperature` nisko, żeby wszyscy widzieli podobny wynik.


In [12]:
import gc
import json
import math
import os
import random
import re
import shutil
import sys
import time
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from getpass import getpass

PYTHON_BIN = Path(sys.executable)
SEED = 1234
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

HF_TOKEN = os.environ.get("HF_TOKEN", "").strip()
if not HF_TOKEN:
    try:
        HF_TOKEN = getpass("Wklej token Hugging Face (read): ").strip()
    except Exception:
        HF_TOKEN = ""
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGINGFACE_HUB_TOKEN"] = HF_TOKEN

DRIVE_OK = False

try:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)
    DRIVE_OK = True
    print("Google Drive mounted.")
except Exception as exc:
    print("Google Drive mount unavailable:", exc)

if DRIVE_OK:
    PERSIST_ROOT = Path("/content/drive/MyDrive/ZZTAI/LAB_E_project_runtime")
else:
    PERSIST_ROOT = Path("/content/zztai_lab_e_project_runtime")
    
DRIVE_OK = False
PERSIST_ROOT = Path("./zztai_lab_e_project_runtime")
PROJECT_ROOT = PERSIST_ROOT / "project2"
DATA_ROOT = PROJECT_ROOT / "data"
RUN_ROOT = PROJECT_ROOT / "runs"
EXPORT_ROOT = PROJECT_ROOT / "exports"
FIGURE_ROOT = PROJECT_ROOT / "figures"

for path in [PROJECT_ROOT, DATA_ROOT, RUN_ROOT, EXPORT_ROOT, FIGURE_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)


Google Drive mount unavailable: No module named 'google.colab'
PROJECT_ROOT: zztai_lab_e_project_runtime/project2


## Cel projektu

Celem jest pokazanie praktycznego workflow dla **małego LLM**:
- jak dostroić go niskokosztowo (`LoRA/QLoRA`),
- jak użyć go do **augmentacji danych**,
- i jak sprawdzić, czy augmentacja poprawia wynik klasyfikatora.

W Lekcji D augmentowaliśmy dane przez **syntetyczne obrazy** z SDXL. Tutaj analogicznie augmentujemy **tekst**.


## Wprowadzenie teoretyczne (skrót)

**LoRA (Low-Rank Adaptation)**:
- zamiast trenować wszystkie wagi modelu, uczymy małe macierze o niskim rangu (`rank=r`),
- to zmniejsza koszt pamięci i czas treningu, a często zachowuje dobrą jakość dostrojenia.

**QLoRA**:
- dodatkowo trzymamy model bazowy w 4-bit (kwantyzacja), a uczymy tylko adapter,
- praktycznie: pozwala trenować większe modele na mniejszym GPU (np. w Colab).

**Augmentacja danych przez LLM**:
- generujemy nowe próbki warunkowane etykietą (np. „klasa: sport” → news o sporcie),
- ryzyka: błędne etykiety, „przeciek” etykiety do tekstu, powielanie szumu, bias.
- dlatego w tej lekcji trzymamy prostą walidację i porównujemy metryki „przed/po”.


# E01 — Przygotowanie danych wejściowych


In [13]:
from datasets import load_dataset

# Używamy publicznego, łatwego zbioru z etykietami.
# (W rozmowie padł pomysł Kaggle „poe short stories”, ale ten zbiór jest zwykle BEZ etykiet,
# więc trudno go użyć do *supervised* klasyfikacji bez dodatkowej anotacji.)
DATASET_ID = "ag_news"

# Zrobimy „mały seed” treningowy, żeby augmentacja miała sens.
SEED_TRAIN_PER_CLASS = 40
SEED_VAL_PER_CLASS = 40
SEED_TEST_PER_CLASS = 200

raw = load_dataset(DATASET_ID)
raw_train = raw["train"]
raw_test = raw["test"]

label_names = raw_train.features["label"].names
num_classes = len(label_names)
print("Classes:", label_names)


Classes: ['World', 'Sports', 'Business', 'Sci/Tech']


In [14]:
def sample_per_class(ds, n_per_class, seed=SEED):
    rng = np.random.default_rng(seed)
    idx_by_label = {i: [] for i in range(num_classes)}
    for i, ex in enumerate(ds):
        idx_by_label[int(ex["label"])].append(i)
    out = []
    for label, idxs in idx_by_label.items():
        idxs = np.array(idxs)
        rng.shuffle(idxs)
        take = idxs[:n_per_class]
        for j in take:
            out.append(ds[int(j)])
    rng.shuffle(out)
    return out

seed_train = sample_per_class(raw_train, SEED_TRAIN_PER_CLASS, seed=SEED)
seed_val = sample_per_class(raw_train, SEED_VAL_PER_CLASS, seed=SEED + 1)
seed_test = sample_per_class(raw_test, SEED_TEST_PER_CLASS, seed=SEED + 2)

seed_train_df = pd.DataFrame(seed_train)
seed_val_df = pd.DataFrame(seed_val)
seed_test_df = pd.DataFrame(seed_test)

display(seed_train_df.head(3))
print('sizes:', len(seed_train_df), len(seed_val_df), len(seed_test_df))


,text,label
0,#39;Dream centre #39; of the brain found Scie...,3
1,"Germany, France support Turkish invitation to ...",2
2,Last season #39;s MVP to miss three games SEAT...,1


sizes: 160 160 800


In [15]:
# Zapis na Drive (żeby potem łatwo budować zbiory i powtarzać eksperymenty)
seed_dir = DATA_ROOT / "seed_ag_news"
seed_dir.mkdir(parents=True, exist_ok=True)
seed_train_df.to_csv(seed_dir / "train.csv", index=False)
seed_val_df.to_csv(seed_dir / "val.csv", index=False)
seed_test_df.to_csv(seed_dir / "test.csv", index=False)

print("Saved:", seed_dir)


Saved: zztai_lab_e_project_runtime/project2/data/seed_ag_news


# E02 — Trening adaptera LoRA/QLoRA na małym LLM


In [17]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from transformers import DataCollatorForLanguageModeling
from transformers import BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Model „pod telefon”: sensowne są 1B–3B param.
# Dobierz model tak, żeby był:
# - publiczny,
# - miał rozsądny tokenizer,
# - dał się później wyeksportować do GGUF.
#
# Przykłady (wybierz jeden):
# - "Qwen/Qwen2.5-1.5B-Instruct"
# - "meta-llama/Llama-3.2-1B-Instruct" (jeśli masz dostęp/licencję na HF)
BASE_MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

USE_4BIT = False  # QLoRA
MAX_LEN = 256

def format_example(label, text):
    # Ważne: NIE wkładaj etykiety do samego tekstu, który ma później iść do klasyfikatora.
    # Etykieta jest tylko warunkiem w promptcie.
    label_name = label_names[int(label)]
    return f"### Klasa: {label_name}\n### Tekst:\n{text.strip()}\n"

train_texts = [format_example(r['label'], r['text']) for r in seed_train]
val_texts = [format_example(r['label'], r['text']) for r in seed_val]

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, use_fast=True, token=HF_TOKEN or None)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize(batch_texts):
    return tokenizer(batch_texts, truncation=True, max_length=MAX_LEN, padding=False)

train_enc = tokenize(train_texts)
val_enc = tokenize(val_texts)

class SimpleListDataset(torch.utils.data.Dataset):
    def __init__(self, enc):
        self.enc = enc
    def __len__(self):
        return len(self.enc['input_ids'])
    def __getitem__(self, idx):
        return {k: torch.tensor(v[idx]) for k, v in self.enc.items()}

train_ds = SimpleListDataset(train_enc)
val_ds = SimpleListDataset(val_enc)

quantization_config = None
if USE_4BIT:
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    token=HF_TOKEN or None,
    device_map="auto",
    quantization_config=quantization_config,
    torch_dtype=torch.float16,
)

if USE_4BIT:
    model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=None,  # auto-detect in PEFT for many HF models
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

run_dir = RUN_ROOT / "lora_ag_news"
run_dir.mkdir(parents=True, exist_ok=True)

import inspect

# Transformers compatibility: some versions use `evaluation_strategy`, others `eval_strategy`.
_ta_params = inspect.signature(TrainingArguments.__init__).parameters
_eval_key = "evaluation_strategy" if "evaluation_strategy" in _ta_params else "eval_strategy"

args = TrainingArguments(
    output_dir=str(run_dir),
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=1,
    logging_steps=10,
    **{_eval_key: 'steps'},
    eval_steps=50,
    save_steps=50,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    report_to=[],
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collator,
)


trainable params: 2,179,072 || all params: 1,545,893,376 || trainable%: 0.1410


/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [18]:
train_result = trainer.train()
metrics = trainer.evaluate()
print(metrics)

adapter_dir = run_dir / "adapter"
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print("Saved adapter:", adapter_dir)


Step,Training Loss,Validation Loss


{'eval_loss': 3.066314697265625, 'eval_runtime': 3.4209, 'eval_samples_per_second': 46.771, 'eval_steps_per_second': 23.386, 'epoch': 1.0}
Saved adapter: zztai_lab_e_project_runtime/project2/runs/lora_ag_news/adapter


In [20]:
# E21 — oddaj ścieżkę do adaptera
# Wymaganie do automatu: ścieżka do folderu na Google Drive (żeby prowadzący mógł łatwo pobrać pliki).
# Jeśli dodatkowo wrzucasz na Hugging Face (extra), możesz wkleić repo w opisie notebooka, ale automat tego nie sprawdza.
final_answer = str(adapter_dir)
wyslij_odpowiedz("E21", "https://github.com/senketsutsu/ZZTAI/tree/main/adapter")


200
Punkty: 0


## Zapis na Hugging Face (opcjonalne, extra; przydatne do iPhone)

Żeby uruchomić model na iPhone bez ręcznego kopiowania plików z Colaba, najprościej:
1) wrzucić wynik do repo na Hugging Face (adapter LoRA, a opcjonalnie też gotowy **GGUF**),
2) pobrać model w aplikacji iOS z integracją z Hugging Face.

Ważne:
- większość aplikacji iOS opartych o `llama.cpp` uruchamia **pełny model w formacie GGUF**, a nie „sam adapter LoRA”.
- dlatego: *adapter upload* jest dobry do archiwizacji i powtarzalności eksperymentu, ale do iPhone zwykle finalnie potrzebujesz sekcji **Eksport GGUF**.

Ta sekcja jest **opcjonalna (extra)** i nie jest wymagana do oceny automatem.

W tej sekcji zapisujemy **adapter LoRA** jako repo (małe pliki). To jest najszybsze i zwykle mieści się w limitach.


In [21]:
from huggingface_hub import login, HfApi

# Jeśli HF_TOKEN jest ustawiony, login powinien przejść bez pytania.
# W przeciwnym razie Colab pokaże prompt do wklejenia tokena.
try:
    login(token=HF_TOKEN or None)
except Exception as exc:
    print("HF login failed:", exc)

#@title E22 - Konfiguracja repo na Hugging Face
HF_REPO_ID = "akrasi/zztai-lab-e-adapter"  #@param {type:"string"}
HF_PRIVATE = True  #@param {type:"boolean"}

assert HF_REPO_ID.strip(), "akrasi/zztai-lab-e-adapter"

api = HfApi()
api.create_repo(repo_id=HF_REPO_ID.strip(), private=HF_PRIVATE, exist_ok=True)
print("Repo ready:", HF_REPO_ID)


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Repo ready: akrasi/zztai-lab-e-adapter


In [22]:
# Upload adaptera (LoRA) do repo
api.upload_folder(
    repo_id=HF_REPO_ID.strip(),
    folder_path=str(adapter_dir),
    path_in_repo="adapter",
    commit_message="Add LoRA adapter (Lab E)",
)

print("Uploaded adapter to:", HF_REPO_ID, "(folder: adapter)")


Upload 2 LFS files:   0%|          | 0/2 [00:00<?, ?it/s]




adapter_model.safetensors: 100%|██████████| 8.73M/8.73M [00:01<00:00, 6.89MB/s]


tokenizer.json: 100%|██████████| 11.4M/11.4M [00:02<00:00, 5.32MB/s]
Upload 2 LFS files: 100%|██████████| 2/2 [00:02<00:00,  1.18s/it]


Uploaded adapter to: akrasi/zztai-lab-e-adapter (folder: adapter)


# E03 — Generowanie syntetycznego zbioru (ok. 50×)


In [28]:
# Imports needed for this section (so you don't have to guess)
import re
import torch
from transformers import AutoModelForCausalLM
from peft import PeftModel, prepare_model_for_kbit_training


# Ładujemy model bazowy + adapter, żeby generować „po dostrojeniu”
base_gen = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    token=HF_TOKEN or None,
    device_map="auto",
    quantization_config=quantization_config,
    torch_dtype=torch.float16,
)
if USE_4BIT:
    base_gen = prepare_model_for_kbit_training(base_gen)
tuned_gen = PeftModel.from_pretrained(base_gen, adapter_dir)
tuned_gen.eval()

@torch.no_grad()
def generate_one(label_name, prompt_seed=None, max_new_tokens=180, do_sample=True, temperature=0.8, top_p=0.95):
    if prompt_seed is None:
        prompt_seed = ""
    prompt = (
        "Wygeneruj krótki news (2-5 zdań) zgodny z klasą. "
        "Nie podawaj nazwy klasy w tekście. "
        f"Klasa: {label_name}.\n"
        f"Seed: {prompt_seed}\n"
        "Tekst:\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(tuned_gen.device)
    out = tuned_gen.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        temperature=temperature,
        top_p=top_p,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
        use_cache=True,
    )
    decoded = tokenizer.decode(out[0], skip_special_tokens=True)
    # wydzielamy tylko część po "Tekst:" jeśli się da
    m = re.split(r"\bTekst:\s*", decoded, maxsplit=1)
    text = m[-1].strip() if m else decoded.strip()
    # prosty cleanup
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Generujemy 50× więcej niż seed_train: per class
AUG_MULTIPLIER = 50
synth_per_class = SEED_TRAIN_PER_CLASS * AUG_MULTIPLIER
print("Synthetic per class:", synth_per_class)

Synthetic per class: 2000


In [29]:
from tqdm import tqdm

In [30]:
total = synth_per_class * len(label_names)
rng = np.random.default_rng(SEED)
synthetic_rows = []
with tqdm(total=total, desc="Generating synthetic data") as pbar:
    for label_idx, label_name in enumerate(label_names):
        for i in range(synth_per_class):
            seed_tag = f"{label_idx}-{i}-{int(rng.integers(0, 1_000_000))}"
            text = generate_one(label_name, prompt_seed=seed_tag, max_new_tokens=80)
            # filtr anty-leak: jeśli model wypisze nazwę klasy, odrzucamy
            if label_name.lower() in text.lower():
                pbar.update(1)
                continue
            if len(text) < 40:
                pbar.update(1)
                continue
            synthetic_rows.append({"label": label_idx, "text": text, "source": "synthetic"})
            pbar.update(1)
        print("class", label_name, "kept", sum(r["label"] == label_idx for r in synthetic_rows))

synth_df = pd.DataFrame(synthetic_rows)
synth_path = DATA_ROOT / "synthetic_ag_news.csv"
synth_df.to_csv(synth_path, index=False)
display(synth_df.sample(3, random_state=SEED))
print("Saved:", synth_path, "rows=", len(synth_df))


Generating synthetic data:  25%|██▌       | 2000/8000 [1:30:09<4:21:37,  2.62s/it]

class World kept 1397


Generating synthetic data:  50%|█████     | 4000/8000 [2:58:55<2:58:04,  2.67s/it]

class Sports kept 1818


Generating synthetic data:  75%|███████▌  | 6000/8000 [4:32:51<1:25:24,  2.56s/it]

class Business kept 1613


Generating synthetic data: 100%|██████████| 8000/8000 [6:03:08<00:00,  2.72s/it]  

class Sci/Tech kept 1985


,label,text,source
6516,3,"A new type of solar cell has been developed, w...",synthetic
2593,1,A young football player scored a goal for his ...,synthetic
486,0,"""Twoje dzieło jest bardzo niezwykłe i zaciekaw...",synthetic


Saved: zztai_lab_e_project_runtime/project2/data/synthetic_ag_news.csv rows= 6813


# E04 — Automatyczne filtrowanie i sanity-check syntetyków


W obrazach z Lekcji D studenci ręcznie odrzucali błędne generacje. Dla tekstu robimy to automatycznie.

Co filtrujemy:
- **label leakage**: jeśli tekst zawiera nazwę klasy (np. „World”, „Sports”) albo jawne metadane „Klasa:”,
- bardzo krótkie lub puste próbki,
- duplikaty (prosta deduplikacja po znormalizowanym tekście).

Celem nie jest „idealne czyszczenie”, tylko sensowny, powtarzalny pipeline.


In [31]:
# Imports needed for this section
import re

def normalize_text(s: str) -> str:
    s = s.lower()
    s = re.sub(r"\s+", " ", s).strip()
    return s

def looks_like_leak(text: str) -> bool:
    t = text.lower()
    if "###" in t or "klasa:" in t or "class:" in t or "label:" in t:
        return True
    for name in label_names:
        if name.lower() in t:
            return True
    return False

before = len(synth_df)
synth_df = synth_df.copy()
synth_df["norm"] = synth_df["text"].map(normalize_text)
synth_df = synth_df[~synth_df["text"].map(looks_like_leak)]
synth_df = synth_df[synth_df["norm"].str.len() >= 40]
synth_df = synth_df.drop_duplicates(subset=["label", "norm"]).drop(columns=["norm"])
after = len(synth_df)

print("Synthetic rows: before =", before, "after =", after, "dropped =", before - after)

synth_filtered_path = DATA_ROOT / "synthetic_ag_news.filtered.csv"
synth_df.to_csv(synth_filtered_path, index=False)
print("Saved:", synth_filtered_path)


Synthetic rows: before = 6813 after = 6514 dropped = 299
Saved: zztai_lab_e_project_runtime/project2/data/synthetic_ag_news.filtered.csv


# E05 — Budowa finalnego zbioru do klasyfikacji


In [32]:
# Imports needed for this section
import pandas as pd
from pathlib import Path

real_train_df = seed_train_df.copy()
real_val_df = seed_val_df.copy()
real_test_df = seed_test_df.copy()

augmented_train_df = pd.concat(
    [
        real_train_df.assign(source="real"),
        synth_df[["label", "text"]].assign(source="synthetic"),
    ],
    ignore_index=True,
)

final_dir = DATA_ROOT / "final_classification_dataset"
final_dir.mkdir(parents=True, exist_ok=True)
augmented_train_df.to_csv(final_dir / "train.csv", index=False)
real_val_df.to_csv(final_dir / "val.csv", index=False)
real_test_df.to_csv(final_dir / "test.csv", index=False)

print("Saved final dataset to:", final_dir)
print("train/val/test sizes:", len(augmented_train_df), len(real_val_df), len(real_test_df))


Saved final dataset to: zztai_lab_e_project_runtime/project2/data/final_classification_dataset
train/val/test sizes: 6674 160 800


# E06 — Klasyfikator i porównanie dokładności


In [33]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

def train_and_eval(train_df, test_df):
    clf = Pipeline(
        [
            ("tfidf", TfidfVectorizer(ngram_range=(1, 2), max_features=50_000)),
            ("lr", LogisticRegression(max_iter=200, n_jobs=1)),
        ]
    )
    clf.fit(train_df["text"].tolist(), train_df["label"].tolist())
    pred = clf.predict(test_df["text"].tolist())
    acc = accuracy_score(test_df["label"].tolist(), pred)
    return acc, pred

real_train_df = seed_train_df.copy()
real_test_df = seed_test_df.copy()
acc_real, pred_real = train_and_eval(real_train_df, real_test_df)
print("Accuracy real-only:", acc_real)


Accuracy real-only: 0.7


In [34]:
# Trening na zbiorze: real seed + synthetic (po filtracji)
acc_aug, pred_aug = train_and_eval(augmented_train_df, real_test_df)
print("Accuracy augmented:", acc_aug)


Accuracy augmented: 0.5975


In [35]:
diff = float(acc_aug - acc_real)

wyslij_odpowiedz("E61", float(acc_real))
wyslij_odpowiedz("E62", float(acc_aug))
wyslij_odpowiedz("E63", diff)


200
Punkty: 0
200
Punkty: 0
200
Punkty: 0


# (Opcjonalnie) Eksport na iPhone: GGUF + aplikacja lokalna


W praktyce najprostsza ścieżka na iPhone dla modeli open-weight to:
1) **scalić** LoRA z bazą (merge),
2) wyeksportować do formatu **GGUF**,
3) zrobić **kwantyzację** (np. `Q4_K_M`),
4) wrzucić GGUF na Hugging Face (opcjonalnie),
5) zaimportować do aplikacji opartej o `llama.cpp`, która wspiera **import własnych modeli** (GGUF) z Files/iCloud lub pobieranie z Hugging Face.

Uwaga:
- nie każda aplikacja na iOS pozwala na import własnych modeli,
- niektóre aplikacje wymagają konkretnego „chat template” dla danego modelu.

Przykładowe aplikacje (stan na 2026-04-14; sprawdź opis w App Store):
- **PocketPal AI** — integracja z Hugging Face i modele GGUF,
- **Enclave** — tryb uruchamiania modeli lokalnych i pobieranie modeli z HF.


In [ ]:
# Ta sekcja jest opcjonalna, bo pobieranie repo i konwersja może potrwać.
DO_GGUF_EXPORT = False  #@param {type:"boolean"}

if DO_GGUF_EXPORT:
    !git clone https://github.com/ggerganov/llama.cpp --depth 1
    !python3 -m pip -q install -U "protobuf<5" "sentencepiece"

    # 1) Merge adapter -> pełne wagi (HF)
    merged_dir = EXPORT_ROOT / "merged_hf"
    merged_dir.mkdir(parents=True, exist_ok=True)

    base_for_merge = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID, token=HF_TOKEN or None, dtype=torch.float16, device_map="cpu"
    )
    tuned_for_merge = PeftModel.from_pretrained(base_for_merge, adapter_dir)
    merged = tuned_for_merge.merge_and_unload()
    merged.save_pretrained(merged_dir)
    tokenizer.save_pretrained(merged_dir)

    # 2) HF -> GGUF
    gguf_f16 = EXPORT_ROOT / "model-f16.gguf"
    !python3 llama.cpp/convert_hf_to_gguf.py {merged_dir} --outtype f16 --outfile {gguf_f16}

    # 3) Quantize
    gguf_q4 = EXPORT_ROOT / "model-q4_k_m.gguf"
    !./llama.cpp/quantize {gguf_f16} {gguf_q4} Q4_K_M

    print("GGUF ready:", gguf_q4)

    # (Opcjonalnie) Upload GGUF do tego samego repo HF (jeśli podano HF_REPO_ID w sekcji wyżej)
    try:
        if "HF_REPO_ID" in globals() and HF_REPO_ID.strip():
            api.upload_file(
                repo_id=HF_REPO_ID.strip(),
                path_or_fileobj=str(gguf_q4),
                path_in_repo="gguf/model-q4_k_m.gguf",
                commit_message="Add GGUF quantized model (Lab E)",
            )
            print("Uploaded GGUF to:", HF_REPO_ID, "(path: gguf/model-q4_k_m.gguf)")
    except Exception as exc:
        print("GGUF upload skipped/failed:", exc)


### Jak uruchomić na iPhone (ogólnie)

Opcja A — Files/iCloud (działa prawie zawsze):
1) Skopiuj plik `model-q4_k_m.gguf` do iCloud Drive lub lokalnie w Files.
2) W aplikacji wybierz „Import model” i wskaż plik GGUF.

Opcja B — pobranie z Hugging Face (wygodne na zajęciach):
1) Upewnij się, że GGUF jest w repo HF (np. `gguf/model-q4_k_m.gguf`).
2) W aplikacji użyj wyszukiwarki Hugging Face i znajdź swoje repo `HF_REPO_ID`.
3) Pobierz GGUF i wybierz go jako aktywny model.

Parametry generacji (żeby studenci mieli podobne wyniki):
- `temperature`: `0.0`–`0.2`
- jeśli aplikacja ma przełącznik „sampling”: wyłącz (`do_sample = False`)
- `top_p`: `1.0` (lub maks)
- `max tokens`: 128–256

Jeśli aplikacja nie ma `do_sample`, to ustaw `temperature=0` i `top_p=1`.

W tej lekcji w części ocenianej używaliśmy ustawień deterministycznych, żeby odpowiedzi studentów były podobne.
